# Notebook 2: TamBERT Tokenizer

Vocabulary size decision = 32k (with unused tokens) [30+2]



## Workflow

| Test | Goal                | Fixed                                                        | Varied              | Options tested                                                                                 | # Runs | Metric                                                 | Depends on              |
| ---- | ------------------- | ------------------------------------------------------------ | ------------------- | ---------------------------------------------------------------------------------------------- | ------ | ------------------------------------------------------ | ----------------------- |
| 1    | Best normalization  | Pre-tokenizer = Whitespace; Algo = BPE; merge count          | Normalization       | NFC vs. None                                                                                   | 2      | OOV, Fertility                                         | —                       |
| 2    | Best pre-tokenizer  | Normalization = Test 1 winner; Algo = BPE; merge count       | Pre-tokenizer stack | (a) Whitespace + codepoint-BPE, (b) Whitespace + grapheme-BPE, (c) Sandhi-split + grapheme-BPE | 3      | OOV, Fertility                                         | Test 1 result           |
| 3    | Best tokenizer algo | Normalization = Test 1 winner; Pre-tokenizer = Test 2 winner | Tokenizer algorithm | BPE, Unigram, WordPiece(BERT), GPE (grapheme-only, reference floor)                            | 4      | OOV, Fertility (+ note GPE is non-comparable baseline) | Test 1 + Test 2 results |

### Metrics

**8. Fertility per source**

$$Fertility = \frac{\text{total subword tokens}}{\text{total whitespace words}}$$

- Target ≤ 2.5 for Tamil; if a specific source spikes (e.g., 4+), it has unusual script mixing or noise.
- SDK: run your trained `BertWordPieceTokenizer` on a sample of each source.


**9. OOV rate per source**

$$OOV\% = \frac{\text{tokens mapped to [UNK]}}{\text{total tokens}} \times 100$$

- Should be < 1% on clean Tamil; a source with 5%+ OOV likely has encoding issues, non-Tamil characters, or very domain-specific jargon.
- SDK: count `[UNK]` in tokenizer output.


---

## Setting up the data

In [1]:
! gdown 1Kzr2Uw8SzEIuvxtYVLJLz6IN-0a73n6V

Downloading...
From (original): https://drive.google.com/uc?id=1Kzr2Uw8SzEIuvxtYVLJLz6IN-0a73n6V
From (redirected): https://drive.google.com/uc?id=1Kzr2Uw8SzEIuvxtYVLJLz6IN-0a73n6V&confirm=t&uuid=8e0a4542-ffa0-4be2-8f44-d773b05eddf9
To: /content/test.txt
100% 1.40G/1.40G [00:15<00:00, 88.6MB/s]


In [2]:
! gdown 1MAHfAon05pVZzHwefIdU8IS8NQK7zL67

Downloading...
From (original): https://drive.google.com/uc?id=1MAHfAon05pVZzHwefIdU8IS8NQK7zL67
From (redirected): https://drive.google.com/uc?id=1MAHfAon05pVZzHwefIdU8IS8NQK7zL67&confirm=t&uuid=c6b0cf28-9baa-4832-a679-6ec5ee6ef99f
To: /content/train.txt
100% 12.7G/12.7G [02:46<00:00, 75.9MB/s]


In [27]:
VOCAB_SIZE = 2000
train_data_path = 'train.txt'
test_data_path = 'test.txt'
metrics_data = 'mertrics.txt'
unk_token = '<unk>'

### Create a small subset for testing

In [4]:
import random

random.seed(42)

def reservoir_sample(input_file, k=5, encoding='utf-8'):
    """
    Performs reservoir sampling to randomly select k lines from the input file.

    Args:
        input_file (str): Path to the input text file.
        k (int): Number of lines to sample (default is 5).
        encoding (str): Encoding of the input file (default is 'utf-8').
    """
    reservoir = []
    with open(input_file, 'r', encoding=encoding) as f:
        for n, line in enumerate(f):
            if n < k:
                reservoir.append(line.strip())
            else:
                j = random.randint(0, n)
                if j < k:
                    reservoir[j] = line.strip()
    return reservoir

Sample the train and test data for a small chunk of lines coz the whole file is too big for experiments

In [6]:
train_data = 'train_data.txt'
test_data = 'test_data.txt'

# Write random files form the train file to metrics file
print(f'Sampling {test_data_path} for 250 lines....')
test_lines = reservoir_sample(test_data_path, 250)
print(f'Sampling {train_data_path} for 1000 lines...')
train_lines = reservoir_sample(train_data_path, 10000)

print(f'Writing to {test_data}...')
# write the test lines to a file
with open(test_data, 'w', encoding='utf-8') as f:
  for line in test_lines:
    f.write(line+'\n')

print(f'Writing to {train_data}...')
# Write lines to training file
with open(train_data, 'w', encoding='utf-8') as f:
  for line in train_lines:
    f.write(line+'\n')

Sampling test.txt for 250 lines
Sampling train.txt for 1000 lines
Writing to test_data.txt...
Writing to train_data.txt...


In [29]:
print(f'Sampling {train_data_path} for 10000 lines...')
train_lines = reservoir_sample(train_data_path, 10000)

print(f'Writing to {train_data}...')
# Write lines to training file
with open(train_data, 'w', encoding='utf-8') as f:
  for line in train_lines:
    f.write(line+'\n')

Sampling train.txt for 1000 lines...
Writing to train_data.txt...


In [7]:
# Initialize variables to be used throughtout the notebook
bpe_special_tokens = ['<unk>','<|endoftext|>']
sample_text = '90 ஒத்த இடத்து நித்திரை கொள்'

In [8]:
from tokenizers import decoders, models, pre_tokenizers, processors, trainers, Tokenizer, normalizers, pre_tokenizers
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import NFC
from tokenizers.models import BPE, Unigram, WordPiece
from tokenizers.trainers import BpeTrainer, UnigramTrainer, WordPieceTrainer

## Metrics

Using this section to setup the OOV and Fertility metrics

In [9]:
def calculate_fertility(tokenizer, file, unk_token):
  unk_id = tokenizer.token_to_id(unk_token)
  if unk_id is None:
      raise ValueError(f"'{unk_token}' not found in tokenizer vocabulary")

  total_words = 0
  total_subword_tokens = 0

  with open(file, 'r', encoding='utf-8') as f:
      for line in f:
          line = line.strip()
          if not line:
              continue
          # Fertility calculation
          total_words += len(line.split())
          total_subword_tokens += len(tokenizer.encode(line).tokens)

  fertility = total_subword_tokens/total_words

  return fertility

In [10]:
def calculate_oov(tokenizer, file, unk_token):
  """
    Method to calculate and return Out-of-Vocabulary % of a tokenizer on a file of text.

    Args:
        tokenizer  - HuggingFace tokenizer (tokenizers-lib style, has .token_to_id / .encode)
        file       - input file path
        unk_token  - the string used for unknown tokens (e.g. "<unk>")

    Returns:
        oov - float, percentage of tokens mapped to unk
  """
  unk_id = tokenizer.token_to_id(unk_token)
  if unk_id is None:
    raise ValueError(f"'{unk_token}' not found in tokenizer vocabulary")

  total_tokens = unk_total = 0

  with open(file, 'r', encoding='utf-8') as f:
    for line in f:
      if not line:
        continue
      token_ids = tokenizer.encode(line).ids
      total_tokens += len(token_ids)
      unk_total += sum(1 for i in token_ids if i == unk_id)

  oov = (unk_total/total_tokens) * 100

## Normalization

Finding the best normalization for tamil text.

For this experiment we will:
- Use huggingface for API
- Set different normailzation methods
- Set same pretokenization method of 'splitting on white space'
- Train a BPE tokenizer each for a VOCAB_SIZE = 1000
- Compare metrics between 2 different normalizations

### No Normalization

In [11]:
%%time

# Define tokenizer with unknown special keyword -> needed for BPE
normal_tokenizer_none = Tokenizer(BPE(unk_token=unk_token))

# Set normalization
normal_tokenizer_none.normalizer = None # first experiment is with NO normalizations

# Set pre-tokenizer
normal_tokenizer_none.pre_tokenizer = pre_tokenizers.Whitespace()

# Visualize how the pre-trainer splits text
print(normal_tokenizer_none.pre_tokenizer.pre_tokenize_str(sample_text))

# Get a trainer for tokenizer
normal_tokenizer_none_trainer = BpeTrainer(vocab_size=VOCAB_SIZE,
                                           special_tokens=bpe_special_tokens)

# Train the tokenizer using trainer
normal_tokenizer_none.train([train_data], trainer=normal_tokenizer_none_trainer)

# Get metrics and test it
normal_tokenizer_none_fertility = calculate_fertility(normal_tokenizer_none, test_data, unk_token)

# Print tokenizer stats
print(f'Tokenizer Vocabulary: {len(normal_tokenizer_none.get_vocab())}')
print(f'Sample sentence: {sample_text}')
print(f'Tokenizer Encoding: {normal_tokenizer_none.encode(sample_text).tokens}')
print(f'Tokenizer Fertility Score {normal_tokenizer_none_fertility}')

[('90', (0, 2)), ('ஒத்த', (3, 7)), ('இடத்து', (8, 14)), ('நித்திரை', (15, 23)), ('கொள்', (24, 28))]
Tokenizer Vocabulary: 1000
Sample sentence: 90 ஒத்த இடத்து நித்திரை கொள்
Tokenizer Encoding: ['9', '0', 'ஒ', 'த்த', 'இட', 'த்து', 'நி', 'த்தி', 'ரை', 'கொ', 'ள்']
Tokenizer Fertility Score 2.598684210526316
CPU times: user 224 ms, sys: 34.7 ms, total: 259 ms
Wall time: 245 ms


### NFC Nromalization

In [12]:
%%time
# Define tokenizer with special keyword
nfc_tokenizer = Tokenizer(BPE(unk_token=unk_token))

# Define normailization
nfc_tokenizer.normalizer = normalizers.NFC()

# Define Pre-tokenizer
nfc_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

# Visualize how the pre-trainer splits text
print(nfc_tokenizer.pre_tokenizer.pre_tokenize_str(sample_text))

# Define trainer for our tokenizer
nfc_trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE,
                                  special_tokens=bpe_special_tokens)

# Train tokenizer
nfc_tokenizer.train([train_data], trainer=nfc_trainer)

# Calculate metrics
nfc_fertility = calculate_fertility(tokenizer=nfc_tokenizer,
                                    file=test_data,
                                    unk_token=unk_token)

# Print the stats
# Print tokenizer stats
print(f'Tokenizer Vocabulary: {len(nfc_tokenizer.get_vocab())}')
print(f'Sample sentence: {sample_text}')
print(f'Tokenizer Encoding: {nfc_tokenizer.encode(sample_text).tokens}')
print(f'Tokenizer Fertility Score {nfc_fertility}')

[('90', (0, 2)), ('ஒத்த', (3, 7)), ('இடத்து', (8, 14)), ('நித்திரை', (15, 23)), ('கொள்', (24, 28))]
Tokenizer Vocabulary: 1000
Sample sentence: 90 ஒத்த இடத்து நித்திரை கொள்
Tokenizer Encoding: ['9', '0', 'ஒ', 'த்த', 'இட', 'த்து', 'நி', 'த்தி', 'ரை', 'கொ', 'ள்']
Tokenizer Fertility Score 2.59688995215311
CPU times: user 229 ms, sys: 29 ms, total: 258 ms
Wall time: 207 ms


In [13]:
# Visualive what the tokenizer does to the text
nfc_tokenizer.normalizer.normalize_str(sample_text)

'90 ஒத்த இடத்து நித்திரை கொள்'

There is no big difference between the fertility scores found. This might be due to the fact that the text is already well preprocessed and all the tet are stored in a consistent form.

## Pre-Tokenizers

This section will explore how different types of pre-tokenization change the fertility score

Follow the same steps, use NFC for normalization, but for pre-tokenization we use these approaches:
- Whitespace + codepoint BPE
- Whitespace + grapheme BPE
- Sandhi script + grapheme BPE

### Codepoint BPE

In [30]:
VOCAB_SIZE

2000

In [31]:
%%time

# Define the tokenizer
codepoint_tokenizer = Tokenizer(BPE(unk_token=unk_token))

# Define normalization
codepoint_tokenizer.normalizer = normalizers.NFC()

# Define pre-tokenizer this time codepoint
codepoint_tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.Whitespace()
])

# Visualize the pre-tokenizer
print(codepoint_tokenizer.pre_tokenizer.pre_tokenize_str(sample_text))

# Define trainer
codepoint_trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE,
                                        special_tokens=bpe_special_tokens)

# Train tokenizer
codepoint_tokenizer.train([train_data], trainer=codepoint_trainer)

# Calculate fertility
codepoint_fertility = calculate_fertility(tokenizer=codepoint_tokenizer,
                                          file=test_data,
                                          unk_token=unk_token)

# Print tokenizer stats
print(f'Tokenizer Vocabulary: {len(codepoint_tokenizer.get_vocab())}')
print(f'Sample sentence: {sample_text}')
print(f'Tokenizer Encoding: {codepoint_tokenizer.encode(sample_text).tokens}')
print(f'Tokenizer Fertility Score {codepoint_fertility}')

[('90', (0, 2)), ('ஒத்த', (3, 7)), ('இடத்து', (8, 14)), ('நித்திரை', (15, 23)), ('கொள்', (24, 28))]
Tokenizer Vocabulary: 2000
Sample sentence: 90 ஒத்த இடத்து நித்திரை கொள்
Tokenizer Encoding: ['9', '0', 'ஒ', 'த்த', 'இட', 'த்து', 'நி', 'த்தி', 'ரை', 'கொள்']
Tokenizer Fertility Score 2.1979665071770333
CPU times: user 1.4 s, sys: 126 ms, total: 1.53 s
Wall time: 1.31 s


### Grapheme BPE

In [15]:
!pip install grapheme

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for grapheme: filename=grapheme-0.6.0-py3-none-any.whl size=210082 sha256=e3314ff0c6cc44eedcecb35b87a4541a8181547d79c4a12553ff9145e12b5b57
  Stored in directory: /root/.cache/pip/wheels/5b/aa/3b/d94434910f5e19ac7f8aa6523d74a46fe06bfcbc7e4b26caf6
Successfully built grapheme


In [ ]:
import grapheme
from tokenizers import NormalizedString, PreTokenizedString

class GraphemePreTokenizer:
  def pre_tokenize(self, pretok=PreTokenizedString):
    def split_graphemes(i, normalized):
      text = str(normalized)
      return [NormalizedString(g) for g in grapheme.graphemes(text)]
    pretok.split(split_graphemes)

Optimized grapheme function using C method for faster processing and corrected splitting

In [16]:
# Fixed grapheme with proper chunking
import regex as re
from tokenizers import NormalizedString, PreTokenizedString

_GRAPHEME_RE = re.compile(r"\X")   # native C-level extended-grapheme-cluster matching

class GraphemePreTokenizer:
    def pre_tokenize(self, pretok: PreTokenizedString):
        def split_graphemes(i, normalized):
            text = str(normalized)
            return [normalized[m.start():m.end()] for m in _GRAPHEME_RE.finditer(text)]
        pretok.split(split_graphemes)

In [32]:
%%time

# Define tokenizer
grapheme_tokenizer = Tokenizer(BPE(unk_token=unk_token))

# Define normalization
grapheme_tokenizer.normalizer = normalizers.NFC()

# Define pre-tokenizer, our custom one
grapheme_tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.Whitespace(),
    pre_tokenizers.PreTokenizer.custom(GraphemePreTokenizer())
])

# Visualize pretokenzier
print(grapheme_tokenizer.pre_tokenizer.pre_tokenize_str(sample_text))

# Define trainer
grapheme_trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE,
                                       special_tokens=bpe_special_tokens)

# Train grapheme tokenizer
grapheme_tokenizer.train([train_data], trainer=grapheme_trainer)

# Calculate metrics
grapheme_fertilty = calculate_fertility(tokenizer=grapheme_tokenizer,
                                        file=test_data,
                                        unk_token=unk_token)

# Print the metircs
print(f'Tokenizer Vocabulary: {len(grapheme_tokenizer.get_vocab())}')
print(f'Sample sentence: {sample_text}')
print(f'Tokenizer Encoding: {grapheme_tokenizer.encode(sample_text).tokens}')
print(f'Tokenizer Fertility Score {grapheme_fertilty}')

[('9', (0, 1)), ('0', (1, 2)), ('ஒ', (3, 4)), ('த்', (4, 6)), ('த', (6, 7)), ('இ', (8, 9)), ('ட', (9, 10)), ('த்', (10, 12)), ('து', (12, 14)), ('நி', (15, 17)), ('த்', (17, 19)), ('தி', (19, 21)), ('ரை', (21, 23)), ('கொ', (24, 26)), ('ள்', (26, 28))]
Tokenizer Vocabulary: 327
Sample sentence: 90 ஒத்த இடத்து நித்திரை கொள்
Tokenizer Encoding: ['9', '0', 'ஒ', 'த்', 'த', 'இ', 'ட', 'த்', 'து', 'நி', 'த்', 'தி', 'ரை', 'கொ', 'ள்']
Tokenizer Fertility Score 4.046351674641149
CPU times: user 5.11 s, sys: 1.58 s, total: 6.69 s
Wall time: 5.81 s


### Sandhi-splite + Grapheme

#### Custome Sandhi splitting code from [link](https://github.com/RoshiniPriya05/Agathiyam-Tamil/blob/main/Agathiyam-%20Sandhi%20aware%20tokenization%20for%20Tamil%20Language/core/sandhi.py)

In [18]:
# First pre-token
# sandhi.py
import regex as re
from dataclasses import dataclass
from typing import List, Tuple

@dataclass
class Rule:
    pattern: re.Pattern
    repl: str

BOUND = "⟂"  # boundary sentinel

# ================================
# Tamil Sandhi rules (yours, preserved)
# ================================

TA_RULES = [

# ---------------------------------------------------------------------
# A) உயிர் + உயிர் (Vowel–Vowel joins) — coalescence & cue marking
# We mark the boundary before the second vowel (or its onset), so merges don’t cross.
# ---------------------------------------------------------------------

# அ + அ/ஆ … (common a+a ā-type joins)
Rule(re.compile(r"(அ)\s*(அ|ஆ)"), r"\1" + BOUND + r"\2"),
# அ + இ/ஈ  → often /e/-like outcome; mark join
Rule(re.compile(r"(அ)\s*(இ|ஈ)"), r"\1" + BOUND + r"\2"),
# அ + உ/ஊ → often /o/-like; mark join
Rule(re.compile(r"(அ)\s*(உ|ஊ)"), r"\1" + BOUND + r"\2"),
# அ + எ/ஏ, ஒ/ஓ, ஐ/ஔ
Rule(re.compile(r"(அ)\s*(எ|ஏ)"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"(அ)\s*(ஒ|ஓ)"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"(அ)\s*(ஐ|ஔ)"), r"\1" + BOUND + r"\2"),

# இ/ஈ + உயிர் (potential y-glide contexts)
Rule(re.compile(r"(இ|ஈ)\s*(அ|ஆ|இ|ஈ|உ|ஊ|எ|ஏ|ஒ|ஓ|ஐ|ஔ)"), r"\1" + BOUND + r"\2"),

# உ/ஊ + உயிர் (potential v-glide contexts)
Rule(re.compile(r"(உ|ஊ)\s*(அ|ஆ|இ|ஈ|உ|ஊ|எ|ஏ|ஒ|ஓ|ஐ|ஔ)"), r"\1" + BOUND + r"\2"),

# எ/ஏ, ஒ/ஓ + உயிர் (diphthong-like joins; keep safe)
Rule(re.compile(r"(எ|ஏ|ஒ|ஓ)\s*(அ|ஆ|இ|ஈ|உ|ஊ|எ|ஏ|ஒ|ஓ|ஐ|ஔ)"), r"\1" + BOUND + r"\2"),

# ஐ/ஔ + உயிர் (mark joins after diphthongs)
Rule(re.compile(r"(ஐ|ஔ)\s*(அ|ஆ|இ|ஈ|உ|ஊ|எ|ஏ|ஒ|ஓ|ஐ|ஔ)"), r"\1" + BOUND + r"\2"),

# ---------------------------------------------------------------------
# B) Glide insertion cues (இடைஎழுத்து தோன்றுதல்) — y/வ positions
# We *mark* the place where a glide typically appears; we don’t insert it.
# Add both independent-vowel and dependent-sign contexts.
# ---------------------------------------------------------------------

# Dependent sign i/ī + அ… (ி/ீ before அ… → y-glide in speech)
Rule(re.compile(r"(ி|ீ)\s*(அ)"), r"\1" + BOUND + r"\2"),
# Dependent sign u/ū + அ… (ு/ூ before அ… → v-glide)
Rule(re.compile(r"(ு|ூ)\s*(அ)"), r"\1" + BOUND + r"\2"),

# Word ends with இ/ஈ, next starts with அ… (independent vowels)
Rule(re.compile(r"(இ|ஈ)\s*(அ)"), r"\1" + BOUND + r"\2"),
# Word ends with உ/ஊ, next starts with அ…
Rule(re.compile(r"(உ|ஊ)\s*(அ)"), r"\1" + BOUND + r"\2"),

# Cases with y/v already present — keep a boundary before the glide
Rule(re.compile(r"(ி|ீ)\s*(ய)"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"(ு|ூ)\s*(வ)"), r"\1" + BOUND + r"\2"),

# ---------------------------------------------------------------------
# C) Nasal + stop assimilations (மெய் சந்தி)
# We DO NOT rewrite to ங்க/ஞ்ச/ண்ட/ந்த/ம்ப; we just mark the join.
# ---------------------------------------------------------------------

# ங் before க/க-series
Rule(re.compile(r"(ங்)\s*(க)"), r"\1" + BOUND + r"\2"),
# ஞ் before ச/ச-series
Rule(re.compile(r"(ஞ்)\s*(ச)"), r"\1" + BOUND + r"\2"),
# ண் before ட/ட-series
Rule(re.compile(r"(ண்)\s*(ட)"), r"\1" + BOUND + r"\2"),
# ந் before த/த-series
Rule(re.compile(r"(ந்)\s*(த)"), r"\1" + BOUND + r"\2"),
# ம் before ப/ப-series
Rule(re.compile(r"(ம்)\s*(ப)"), r"\1" + BOUND + r"\2"),
# ன் before ந
Rule(re.compile(r"(ன்)\s*(ந)"), r"\1" + BOUND + r"\2"),

# Generic nasal + stop cluster (safety net)
Rule(re.compile(r"(ங்|ஞ்|ண்|ந்|ம்|ன்)\s*(க|ச|ட|த|ப|ற)"), r"\1" + BOUND + r"\2"),

# ---------------------------------------------------------------------
# D) Gemination / doubling across boundary (compounds)
# ---------------------------------------------------------------------

Rule(re.compile(r"(க்)\s*(க)"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"(ச்)\s*(ச)"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"(ட்)\s*(ட)"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"(த்)\s*(த)"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"(ப்)\s*(ப)"), r"\1" + BOUND + r"\2"),

# Liquids/approximants doubling across boundary
Rule(re.compile(r"(ய்)\s*(ய)"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"(வ்)\s*(வ)"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"(ல்)\s*(ல)"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"(ள்)\s*(ள)"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"(ர்)\s*(ர)"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"(ற்)\s*(ற)"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"(ன்)\s*(ன)"), r"\1" + BOUND + r"\2"),

# ---------------------------------------------------------------------
# E) திரிதல் (mutation) cues — mark classic change environments
# ---------------------------------------------------------------------

# ல் + ச
Rule(re.compile(r"(ல்)\s*(ச)"), r"\1" + BOUND + r"\2"),
# ர்/ற் + ர
Rule(re.compile(r"(ர்)\s*(ர)"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"(ற்)\s*(ர)"), r"\1" + BOUND + r"\2"),
# Dental↔retroflex interplay triggers
Rule(re.compile(r"(ன்|ண்)\s*(ட|த)"), r"\1" + BOUND + r"\2"),

# ---------------------------------------------------------------------
# F) கெடுதல் (final consonant loss before vowel) — mark likely joins
# ---------------------------------------------------------------------

Rule(re.compile(r"(க்)\s*([அஆஇஈஉஊஎஏஒஓஐஔ])"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"(ச்)\s*([அஆஇஈஉஊஎஏஒஓஐஔ])"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"(ட்)\s*([அஆஇஈஉஊஎஏஒஓஐஔ])"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"(த்)\s*([அஆஇஈஉஊஎஏஒஓஐஔ])"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"(ப்)\s*([அஆஇஈஉஊஎஏஒஓஐஔ])"), r"\1" + BOUND + r"\2"),

# Final sonorants often reduce/elide before suffix vowels
Rule(re.compile(r"(ம்)\s*([அஆஇஈஉஊஎஏஒஓஐஔ])"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"(ய்)\s*([அஆஇஈஉஊஎஏஒஓஐஔ])"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"(ல்)\s*([அஆஇஈஉஊஎஏஒஓஐஔ])"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"(ள்)\s*([அஆஇஈஉஊஎஏஒஓஐஔ])"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"(ர்)\s*([அஆஇஈஉஊஎஏஒஓஐஔ])"), r"\1" + BOUND + r"\2"),

# ---------------------------------------------------------------------
# G) Case-suffix & postposition joins (வேற்றுமைச் சந்தி) — frequent cues
# ---------------------------------------------------------------------

Rule(re.compile(r"([அஆஇஈஉஊஎஏஒஓஐஔ])\s*(ஐ)"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"([அஆஇஈஉஊஎஏஒஓஐஔ])\s*((உ|க்)கு)"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"([அஆஇஈஉஊஎஏஒஓஐஔ])\s*(ஆல்|னால்)"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"([அஆஇஈஉஊஎஏஒஓஐஔ])\s*(இல்|அல்)"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"([அஆஇஈஉஊஎஏஒஓஐஔ])\s*(இடம்|உடன்|முன்|பின்)"), r"\1" + BOUND + r"\2"),

# Noun + plural/collective markers
Rule(re.compile(r"([அஆஇஈஉஊஎஏஒஓஐஔ])\s*(கள்)"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"([அஆஇஈஉஊஎஏஒஓஐஔ])\s*(வர்|வர்கள்)"), r"\1" + BOUND + r"\2"),

# ---------------------------------------------------------------------
# H) Verbal participle and auxiliary joins (எச்சம்/வினைச் சந்தி)
# ---------------------------------------------------------------------

# -இ/-உ/-அ participles + போ/வா/இரு/உள்…
Rule(re.compile(r"(ி|உ|அ)\s*(போ|வா|இரு|உள்)"), r"\1" + BOUND + r"\2"),

# -த்து/-ட்டு + auxiliary
Rule(re.compile(r"(த்து|ட்டு)\s*(கொள்|விடு|போ|ஆகு)"), r"\1" + BOUND + r"\2"),

# -ஆன/-என்/-உம் adjectival/relativizer + noun
Rule(re.compile(r"(ஆன|என்|உம்)\s*([அஆஇஈஉஊஎஏஒஓஐஔஅ-ஹ])"), r"\1" + BOUND + r"\2"),

# ---------------------------------------------------------------------
# I) Numeral + classifier/suffix
# ---------------------------------------------------------------------

Rule(re.compile(r"([௦-௯0-9]+)\s*(ஆம்(?: நாள்| ஆண்டு)?)"), r"\1" + BOUND + r"\2"),
Rule(re.compile(r"([௦-௯0-9]+)\s*(ஐ)"), r"\1" + BOUND + r"\2"),

# ---------------------------------------------------------------------
# J) Generic whitespace suppression inside compounds
# ---------------------------------------------------------------------

Rule(re.compile(r"(\S)\s+(?=\S)"), r"\1" + BOUND),
]

# English: pass-through (no phonological sandhi)
EN_RULES: List[Rule] = []

LANG_RULES = {
    "ta": TA_RULES,
    "tamil": TA_RULES,   # alias
    "en": EN_RULES,
    "english": EN_RULES,
}

# ---------- Helpers for code-mixed handling ----------

TAMIL_RANGE = r"\u0B80-\u0BFF"
RE_TAMIL = re.compile(fr"[{TAMIL_RANGE}]")
RE_WORD_OR_SPACE_OR_PUNC = re.compile(r"\w+|\s+|[^\w\s]")

def apply_rules(text: str, rules: List[Rule]) -> str:
    out = text
    for r in rules:
        out = r.pattern.sub(r.repl, out)
    return out

def sandhi_mark(text: str, lang="ta"):
    rules = LANG_RULES.get(lang, [])
    return apply_rules(text, rules)

def _mark_mixed(text: str) -> str:
    """
    Apply Tamil sandhi rules only to Tamil spans; leave non-Tamil spans as-is.
    This ensures English/Tanglish chunks don't get Tamil-specific boundaries.
    """
    chunks = RE_WORD_OR_SPACE_OR_PUNC.findall(text)
    out_parts = []
    for ch in chunks:
        if RE_TAMIL.search(ch):
            out_parts.append(sandhi_mark(ch, "ta"))
        else:
            # English/Latin/digits/punct/spaces -> no sandhi rules
            out_parts.append(sandhi_mark(ch, "en"))  # pass-through
    return "".join(out_parts)

def sandhi_split(text: str, lang="ta") -> List[Tuple[str, Tuple[int,int]]]:
    """
    Returns [(token, (start,end))] splitting on BOUND after applying rules.
    Keeps offsets relative to the *post-rule* string.
    - lang="ta" -> Tamil rules
    - lang="en" -> pass-through
    - lang="mix" -> per-span Tamil-only marking
    """
    if lang.lower() in ("mix", "code-mix", "codemix", "cmix"):
        marked = _mark_mixed(text)
    else:
        marked = sandhi_mark(text, lang)

    parts = marked.split(BOUND)
    tokens = []
    cursor = 0
    for part in parts:
        for w in re.findall(r"\S+|\s+", part):
            tokens.append((w, (cursor, cursor+len(w))))
            cursor += len(w)
    return tokens

def remove_boundaries(text: str) -> str:
    return text.replace(BOUND, "")

In [19]:
from typing import Set  # add to your existing typing import line

# ============================================================
# FAST PATH — no string rewriting; offsets are always exact
# into the original `text`. Replaces sandhi_mark + _mark_mixed
# + the old BOUND-split logic.
# ============================================================

# All non-J rules are (group1)...(group2) with intent "boundary at
# group 2's start" — that's what \1 + BOUND + \2 meant. Rule J (the
# last entry, whitespace suppression) is excluded: whitespace
# boundaries are now computed directly and correctly below instead.
TA_PHONOLOGICAL_RULES: List[Rule] = TA_RULES[:-1]

def _rule_boundary(m) -> int:
    if m.lastindex and m.lastindex >= 2:
        return m.start(2)
    return m.end()

def _phonological_boundaries(text: str, rules: List[Rule]) -> Set[int]:
    bounds: Set[int] = set()
    for r in rules:
        for m in r.pattern.finditer(text):   # scan only, no .sub(), no rewriting
            bounds.add(_rule_boundary(m))
    return bounds

_WS_SPLIT_RE = re.compile(r"\S+|\s+")

def _whitespace_boundaries(text: str) -> Set[int]:
    # Every \S<->\s transition. Correct, non-destructive replacement for rule J.
    return {m.start() for m in _WS_SPLIT_RE.finditer(text)}

def sandhi_split(text: str, lang: str = "ta") -> List[Tuple[str, Tuple[int, int]]]:
    """
    Same signature and return shape as before: [(token, (start, end)), ...].
    Guarantees text[start:end] == token for every tuple — the old version did not.
    """
    lang = lang.lower()
    if lang in ("en", "english"):
        rules: List[Rule] = []
    else:
        # "ta"/"tamil"/"mix"/"cmix" all use the same rules — every TA_RULES
        # pattern only matches Tamil-range characters, so scanning a mixed
        # string directly is safe; no chunk pre-splitting needed.
        rules = TA_PHONOLOGICAL_RULES

    bounds = _whitespace_boundaries(text)
    if rules:
        bounds |= _phonological_boundaries(text, rules)
    bounds.add(0)
    bounds.add(len(text))

    cut_points = sorted(bounds)
    tokens: List[Tuple[str, Tuple[int, int]]] = []
    for start, end in zip(cut_points, cut_points[1:]):
        if end > start:
            tokens.append((text[start:end], (start, end)))
    return tokens

#### Covert the Sandhit spliter into a valide Pre-tokenizer

In [ ]:
class SandhiPreTokenizer:
  def pre_tokenize(self, pretok: PreTokenizedString):
    def split_sandhi(i, normalized):
      text = str(normalized)
      chuncks = sandhi_split(text, lang='ta')
      return [NormalizedString(tok) for tok, _ in chuncks]
    pretok.split(split_sandhi)

In [20]:
# Corrected version of SandhiPreTokenizer
class SandhiPreTokenizer:
    def pre_tokenize(self, pretok: PreTokenizedString):
        def split_sandhi(i, normalized):
            text = str(normalized)
            chunks = sandhi_split(text, lang="ta")
            pieces = []
            for tok, (start, end) in chunks:
                pieces.append(normalized[start:end])  # slice using sandhi_split's own offsets
            return pieces
        pretok.split(split_sandhi)

#### Train tokenizer

In [33]:
%%time

# Define tokenizer
sandhi_tokenizer = Tokenizer(BPE(unk_token=unk_token))

# Define normalizers
sandhi_tokenizer.normalizer = normalizers.NFC()

# Define pre-tokenizers apply the new split on top of grapheme
sandhi_tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.PreTokenizer.custom(SandhiPreTokenizer()),
    pre_tokenizers.PreTokenizer.custom(GraphemePreTokenizer())
])

# Visualize pretokenizer transformation
print(sandhi_tokenizer.pre_tokenizer.pre_tokenize_str(sample_text))

# Define trainer
sandhi_trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE,
                                     special_tokens=bpe_special_tokens)

# Train the tokenizer
sandhi_tokenizer.train([train_data], trainer=sandhi_trainer)

# Calculate metrics
sandhi_fertility = calculate_fertility(tokenizer=sandhi_tokenizer,
                                       file=test_data,
                                       unk_token=unk_token)

# Print the tokenizer statistics
# Print the metircs
print(f'Tokenizer Vocabulary: {len(sandhi_tokenizer.get_vocab())}')
print(f'Sample sentence: {sample_text}')
print(f'Tokenizer Encoding: {sandhi_tokenizer.encode(sample_text).tokens}')
print(f'Tokenizer Fertility Score {sandhi_fertility}')

[('9', (0, 1)), ('0', (1, 2)), (' ', (2, 3)), ('ஒ', (3, 4)), ('த்', (4, 6)), ('த', (6, 7)), (' ', (7, 8)), ('இ', (8, 9)), ('ட', (9, 10)), ('த்', (10, 12)), ('து', (12, 14)), (' ', (14, 15)), ('நி', (15, 17)), ('த்', (17, 19)), ('தி', (19, 21)), ('ரை', (21, 23)), (' ', (23, 24)), ('கொ', (24, 26)), ('ள்', (26, 28))]
Tokenizer Vocabulary: 329
Sample sentence: 90 ஒத்த இடத்து நித்திரை கொள்
Tokenizer Encoding: ['9', '0', ' ', 'ஒ', 'த்', 'த', ' ', 'இ', 'ட', 'த்', 'து', ' ', 'நி', 'த்', 'தி', 'ரை', ' ', 'கொ', 'ள்']
Tokenizer Fertility Score 4.971590909090909
CPU times: user 12.2 s, sys: 3.54 s, total: 15.8 s
Wall time: 15.8 s


### Sandhi-splite + Codepoint

In [34]:
%%time

# Define tokenizer
sandhi_tokenizer = Tokenizer(BPE(unk_token=unk_token))

# Define normalizers
sandhi_tokenizer.normalizer = normalizers.NFC()

# Define pre-tokenizers apply the new split on top of grapheme
sandhi_tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.PreTokenizer.custom(SandhiPreTokenizer())
])

# Visualize pretokenizer transformation
print(sandhi_tokenizer.pre_tokenizer.pre_tokenize_str(sample_text))

# Define trainer
sandhi_trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE,
                                     special_tokens=bpe_special_tokens)

# Train the tokenizer
sandhi_tokenizer.train([train_data], trainer=sandhi_trainer)

# Calculate metrics
sandhi_fertility = calculate_fertility(tokenizer=sandhi_tokenizer,
                                       file=test_data,
                                       unk_token=unk_token)

# Print the tokenizer statistics
# Print the metircs
print(f'Tokenizer Vocabulary: {len(sandhi_tokenizer.get_vocab())}')
print(f'Sample sentence: {sample_text}')
print(f'Tokenizer Encoding: {sandhi_tokenizer.encode(sample_text).tokens}')
print(f'Tokenizer Fertility Score {sandhi_fertility}')

[('90', (0, 2)), (' ', (2, 3)), ('ஒத்', (3, 6)), ('த', (6, 7)), (' ', (7, 8)), ('இடத்', (8, 12)), ('து', (12, 14)), (' ', (14, 15)), ('நித்', (15, 19)), ('திரை', (19, 23)), (' ', (23, 24)), ('கொள்', (24, 28))]
Tokenizer Vocabulary: 2000
Sample sentence: 90 ஒத்த இடத்து நித்திரை கொள்
Tokenizer Encoding: ['9', '0', ' ', 'ஒத்', 'த', ' ', 'இடத்', 'து', ' ', 'நி', 'த்', 'திரை', ' ', 'கொள்']
Tokenizer Fertility Score 3.325956937799043
CPU times: user 6.74 s, sys: 1.49 s, total: 8.23 s
Wall time: 10.5 s


## Train tokenizers

### Unigram Tokenizer

In [35]:
from tokenizers import decoders, models, normalizers, pre_tokenizers, processors, trainers, Tokenizer

tokenizer = Tokenizer(models.Unigram())

In [36]:
# Disable normalization
tokenizer.normalizer = normalizers.Sequence([
    normalizers.Replace("``", '"'),
    normalizers.Replace("''", '"'),
])

# Setup pre-tokenizer
tokenizer.pre_tokenizer = pre_tokenizers.Metaspace()

Preprocessed look of a sample sentence

In [37]:
# Test the processing pipeline
sample_text = '90 ஒத்த இடத்து நித்திரை கொள்'
tokenizer.pre_tokenizer.pre_tokenize_str(sample_text)

[('▁90', (0, 2)),
 ('▁ஒத்த', (2, 7)),
 ('▁இடத்து', (7, 14)),
 ('▁நித்திரை', (14, 23)),
 ('▁கொள்', (23, 28))]

In [38]:
special_tokens = ["[CLS]", "[SEP]", "<unk>", "<pad>", "[MASK]"]

# Need to set the special tokens
trainer = trainers.UnigramTrainer(vocab_size=VOCAB_SIZE,
                                  unk_token='<unk>',
                                  special_tokens=special_tokens)


In [42]:
# Train the tokenizer
tokenizer.train([train_data], trainer=trainer)

In [43]:
cls_token_id = tokenizer.token_to_id('[CLS]')
sep_token_id = tokenizer.token_to_id('[SEP]')

In [44]:
tokenizer.post_processor = processors.TemplateProcessing(
    single="[CLS]:0 $A:0 [SEP]:0",
    pair="[CLS]:0 $A:0 [SEP]:0 $B:1 [SEP]:1",
    special_tokens=[
        ("[CLS]", cls_token_id),
        ("[SEP]", sep_token_id),
    ],
)
tokenizer.decoder = decoders.Metaspace()

In [45]:
from transformers import AlbertTokenizerFast

new_tokenizer = AlbertTokenizerFast(tokenizer_object=tokenizer)

### BPE Tokenizer

In [46]:
tokenizer = Tokenizer(models.BPE(unk_token='[UNK]'))

In [47]:
# Add normalization and pre-tokenizers
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

In [48]:
# View how a sample is tokenized
tokenizer.pre_tokenizer.pre_tokenize_str(sample_text)

[('90', (0, 2)),
 ('ஒத்த', (3, 7)),
 ('இடத்து', (8, 14)),
 ('நித்திரை', (15, 23)),
 ('கொள்', (24, 28))]

In [49]:
# Add spcial tokens
bpe_special_tokens = '<|endoftext|>'

trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE,
                              special_tokens=[bpe_special_tokens])

In [50]:
%%time

# train the tokenizer
tokenizer.train([train_data], trainer=trainer)

CPU times: user 1.31 s, sys: 51.6 ms, total: 1.36 s
Wall time: 1.57 s


In [51]:
# Check sample encoding
tokenizer.encode(sample_text).tokens

['9', '0', 'ஒ', 'த்த', 'இட', 'த்து', 'நி', 'த்தி', 'ரை', 'கொள்']

In [52]:
vocab = tokenizer.get_vocab()
len(vocab)

2000

In [53]:
# Wrap up in tokenizer object
from transformers import GPT2TokenizerFast
new_tokenizer = GPT2TokenizerFast(tokenizer_object=tokenizer)

In [54]:
# Post processing
tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)
tokenizer.decoder = decoders.ByteLevel()


### WordPiece (BERT) Tokenizer

In [57]:
%%time
# Define tokenizer
tokenizer = Tokenizer(WordPiece(unk_token=unk_token))

# Define normalization
tokenizer.normalizer = normalizers.NFC()

# Define pre-tokenizer
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

special_tokens = ["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"]
trainer = trainers.WordPieceTrainer(vocab_size=VOCAB_SIZE, special_tokens=special_tokens)

# train the tokenizer
tokenizer.train([train_data], trainer=trainer)

cls_token_id = tokenizer.token_to_id("[CLS]")
sep_token_id = tokenizer.token_to_id("[SEP]")
print(cls_token_id, sep_token_id)

tokenizer.post_processor = processors.TemplateProcessing(
    single=f"[CLS]:0 $A:0 [SEP]:0",
    pair=f"[CLS]:0 $A:0 [SEP]:0 $B:1 [SEP]:1",
    special_tokens=[
        ("[CLS]", cls_token_id),
        ("[SEP]", sep_token_id),
    ],
)

CPU times: user 1.5 s, sys: 31.2 ms, total: 1.53 s
Wall time: 1.52 s


In [58]:
# Check sample encoding
tokenizer.encode(sample_text).tokens

['9', '##0', 'ஒ', '##த்த', 'இட', '##த்து', 'நி', '##த்திர', '##ை', 'கொள்']

In [59]:
from transformers import BertTokenizerFast

new_tokenizer = BertTokenizerFast(tokenizer_object=tokenizer)

**Note:** Flow of the tokenizer:

Normalizer standardizes text → pre-tokenizer performs one coarse split into word-like chunks (with markers like ▁ embedded for reversibility) → the model applies its already-trained, frozen merge rules to split those chunks into subwords from a fixed vocabulary (no new learning happens per-sentence) → the post-processor injects architecture-specific structural tokens ([CLS]/[SEP] for BERT, <|endoftext|> for GPT, token-type IDs for sentence pairs) → the decoder reverses the token representation, using the pre-tokenizer's own markers as a guide, back into clean human-readable text.